<a href="https://colab.research.google.com/github/yosie111/rag_shul/blob/main/Copy_of_Step_0_Text_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 1 — Step 0: Noise Cleaning

**Input:** `Shulchan_Aruch_Text.txt`  
**Output:** `Shulchan_Aruch_Cleaned.txt`

### Cleaning Operations
| # | Operation | Details |
|---|-----------|--------|
| 1 | Remove HTML | Tags |
| 2 | Remove zero-width characters | BOM, ZWSP, ZWNJ, ZWJ |
| 3 | Remove underscore lines | Separator lines |
| 4 | Normalize spaces | Multiple spaces → single space |
| 5 | Normalize blank lines | 3+ blank lines → 2 |
| A | Remove duplicate TOC | 736 duplicate heading lines |
| B | Blank lines within siman | Unnecessary gaps between sections |
| C | Space before period | 4 occurrences |
| D | Space before closing paren | 2 occurrences |
| E | Orphan line `1.` | Merge into `סעיף 1.` |
| F | Double-quotes `"` | Convert to `''` |

## Installation

In [1]:
!pip install -q tqdm

## Settings and Imports

In [2]:
import os, re, json, time, sys
from tqdm.auto import tqdm
from collections import defaultdict
from google.colab import files as colab_files
from IPython.display import display, HTML

def hprint(msg):
    display(HTML(
        f'<div dir="rtl" style="text-align:right; '
        f'font-family:monospace; white-space:pre-wrap; '
        f'font-size:14px; line-height:1.8;">{msg}</div>'
    ))

def hprint_table(title, rows):
    html = f'<div dir="rtl" style="text-align:right;">'
    html += f'<h4 style="margin:8px 0;">{title}</h4>'
    html += '<table style="border-collapse:collapse; font-family:monospace; font-size:14px;">'
    first = True
    for cells in rows:
        html += '<tr>'
        for cell in cells:
            tag = 'th' if first else 'td'
            bdr = 'border-bottom:2px solid #333; font-weight:bold;' if first else 'border-bottom:1px solid #ddd;'
            html += f'<{tag} style="padding:4px 16px 4px 0; {bdr}">{cell}</{tag}>'
        html += '</tr>'
        first = False
    html += '</table></div>'
    display(HTML(html))

CORPUS_PATH  = 'Shulchan_Aruch_Text.txt'
CLEANED_PATH = 'Shulchan_Aruch_Cleaned.txt'

_HEB_VALS = {
    'א':1,'ב':2,'ג':3,'ד':4,'ה':5,'ו':6,'ז':7,'ח':8,'ט':9,
    'י':10,'כ':20,'ל':30,'מ':40,'נ':50,'ס':60,'ע':70,'פ':80,'צ':90,
    'ק':100,'ר':200,'ש':300,'ת':400,
}

def heb_to_int(s):
    s = s.replace('״','').replace('\uff02','').replace("''",'')
    return sum(_HEB_VALS.get(ch, 0) for ch in s)

SIMAN_RE = re.compile(r'^סימן\s+([\dא-ת״\uff02]+)\s*-', re.MULTILINE)

def parse_siman_id(raw):
    raw = raw.strip()
    if raw.isdigit(): return int(raw)
    return heb_to_int(raw)

hprint('✅ הגדרות וכלים נטענו')
hprint(f'   בדיקת heb_to_int:  א={heb_to_int("א")}  כג={heb_to_int("כג")}  תרצז={heb_to_int("תרצז")}')

## Load Corpus

In [3]:
if not os.path.exists(CORPUS_PATH):
    hprint('העלה קובץ:')
    uploaded = colab_files.upload()
    if uploaded:
        name = list(uploaded.keys())[0]
        if name != CORPUS_PATH:
            os.rename(name, CORPUS_PATH)
assert os.path.exists(CORPUS_PATH), f'הקובץ {CORPUS_PATH} לא נמצא!'
with open(CORPUS_PATH, encoding='utf-8') as f:
    raw_text = f.read()
hprint(f'✅ נטען: {len(raw_text):,} תווים')

Saving Shulchan_Aruch_Text.txt to Shulchan_Aruch_Text.txt


## Quick Check — Is the Corpus Valid?

In [4]:
siman_matches = SIMAN_RE.findall(raw_text)
rows = [
    ['מדד', 'ערך'],
    ['גודל טקסט', f'{len(raw_text):,} תווים'],
    ['סימנים (regex)', f'{len(siman_matches)} תוצאות'],
]
if siman_matches:
    rows.append(['ראשון', f'סימן {siman_matches[0]}'])
    rows.append(['אחרון', f'סימן {siman_matches[-1]}'])
hprint_table('בדיקת קורפוס', rows)

מדד,ערך
גודל טקסט,"1,061,906 תווים"
סימנים (regex),1394 תוצאות
ראשון,סימן 1
אחרון,סימן 697


## Step 0 — Noise Cleaning

`clean_noise` — Basic cleaning (1–5)  
`clean_extra` — Additional improvements (A–F)

In [5]:
def clean_noise(text):
    text = re.sub(r'<[^>]+>', '', text)
    text = text.replace('\ufeff', '').replace('\u200b', '')
    text = text.replace('\u200c', '').replace('\u200d', '')
    text = re.sub(r'^_+\s*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r' +\n', '\n', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'^\s+$', '', text, flags=re.MULTILINE)
    return text.strip()


def clean_extra(text):
    counters = {}
    lines = text.split('\n')

    # -- A: Remove duplicate TOC --
    content_start = 0
    for i, line in enumerate(lines):
        if re.match(r'^(סעיף\s|\d+\.\s|הגה:)', line):
            content_start = i
            break
    toc_end = content_start
    for j in range(content_start - 1, -1, -1):
        if lines[j].strip().startswith('הלכות '):
            toc_end = j
            break
    content_set = set(l.strip() for l in lines[toc_end:])
    new_lines = []
    toc_removed = 0
    for i, line in enumerate(lines):
        if i < toc_end:
            s = line.strip()
            if s == '' or s in content_set:
                toc_removed += 1
                continue
        new_lines.append(line)
    lines = new_lines
    counters['A: שורות TOC שהוסרו'] = toc_removed

    # -- B: Remove unnecessary blank lines --
    result = []
    blank_removed = 0
    for i, line in enumerate(lines):
        if line.strip() == '':
            j = i + 1
            while j < len(lines) and lines[j].strip() == '':
                j += 1
            if j < len(lines):
                nxt = lines[j].strip()
                if nxt.startswith('סימן ') or nxt.startswith('הלכות '):
                    result.append(line)
                    continue
            blank_removed += 1
            continue
        result.append(line)
    counters['B: שורות ריקות שהוסרו'] = blank_removed
    text = '\n'.join(result)

    # -- C: Space before punctuation --
    c_count = len(re.findall(r' +[.:;,]', text))
    text = re.sub(r' +([.:;,])', r'\1', text)
    counters['C: רווח לפני פיסוק'] = c_count

    # -- D: Space before closing parenthesis --
    d_count = len(re.findall(r' +\)', text))
    text = re.sub(r' +\)', ')', text)
    counters['D: רווח לפני סוגר'] = d_count

    # -- E: Orphan line "N." --
    e_count = len(re.findall(r'^\d+\.$\n', text, re.MULTILINE))
    text = re.sub(r'^(\d+)\.$\n', r'סעיף \1. ', text, flags=re.MULTILINE)
    counters['E: שורות יתומות'] = e_count

    return text, counters


hprint('✅ clean_noise + clean_extra מוכנות')

## Run Step 0 + Report

In [6]:
after_basic = clean_noise(raw_text)
cleaned_text, counters = clean_extra(after_basic)

len_raw     = len(raw_text)
len_final   = len(cleaned_text)
lines_raw   = raw_text.count('\n') + 1
lines_final = cleaned_text.count('\n') + 1
blank_raw   = sum(1 for l in raw_text.split('\n')   if l.strip() == '')
blank_final = sum(1 for l in cleaned_text.split('\n') if l.strip() == '')
siman_before = len(SIMAN_RE.findall(raw_text))
siman_after  = len(SIMAN_RE.findall(cleaned_text))

hprint_table('דוח שלב 0 — ניקוי רעשים', [
    ['מדד',         'לפני',            'אחרי',             'הפרש'],
    ['תווים',       f'{len_raw:,}',     f'{len_final:,}',      f'{len_raw - len_final:,}'],
    ['שורות',       f'{lines_raw:,}',   f'{lines_final:,}',    f'{lines_raw - lines_final:,}'],
    ['שורות ריקות', f'{blank_raw:,}',    f'{blank_final:,}',     f'{blank_raw - blank_final:,}'],
    ['סימנים',      f'{siman_before}',   f'{siman_after}',       ''],
])

detail_rows = [['שיפור', 'מספר תיקונים']]
for key, val in counters.items():
    detail_rows.append([key, str(val)])
hprint_table('פירוט שיפורים', detail_rows)

if siman_after == siman_before:
    hprint('✅ מספר הסימנים לא השתנה — הניקוי תקין!')
elif siman_after == siman_before // 2:
    hprint(f'✅ הסימנים ירדו לחצי (מ-{siman_before} ל-{siman_after}) — TOC הוסר בהצלחה!')
else:
    hprint(f'⚠ שינוי במספר סימנים: לפני={siman_before}, אחרי={siman_after}')

מדד,לפני,אחרי,הפרש
תווים,"1,061,906","1,023,474","38,432"
שורות,"9,036","7,289","1,747"
שורות ריקות,"1,745",735,"1,010"
סימנים,1394,697,


שיפור,מספר תיקונים
A: שורות TOC שהוסרו,814
B: שורות ריקות שהוסרו,931
C: רווח לפני פיסוק,2
D: רווח לפני סוגר,2
E: שורות יתומות,1


## Step 0F — Unify Quotation Marks (Strategy A)

The text uses two different symbols for quotation marks within abbreviations:
- `''` (double apostrophe, 2×U+0027) — **2,125** occurrences (99.4%)
- `"` (double-quote, U+0022) — **13** occurrences (0.6%)

### Why Strategy A?
- **Meaning:** Fully preserved — רש''י remains רש''י
- **Tokenizer:** Apostrophe is standard ASCII — every tokenizer recognizes it
- **Minimal risk:** Only 13 occurrences change, the rest are already in the correct format

A comparison of all 5 strategies (A–E) will be done in Step 2 (abbreviation normalization).

In [7]:
# ── State before unification ──
dq_before   = cleaned_text.count('"')
apos_before = cleaned_text.count("''")

hprint_table('מצב לפני איחוד', [
    ['סימן', 'קוד', 'מספר', 'אחוז'],
    ["''", '2\u00d7U+0027', f'{apos_before}', '99.4%'],
    ['"',  'U+0022',   f'{dq_before}',   '0.6%'],
])

# ── Show all cases that will change ──
if dq_before > 0:
    cases = re.findall(r'[\u05d0-\u05ea]+"[\u05d0-\u05ea]+', cleaned_text)
    from collections import Counter
    c = Counter(cases)
    case_rows = [['מקור', 'מספר', 'תוצאה']]
    for word, cnt in c.most_common():
        fixed = word.replace('"', "''")
        case_rows.append([word, str(cnt), fixed])
    hprint_table(f'מקרים שישתנו ({dq_before} מופעים)', case_rows)
else:
    hprint('✅ אין מרכאות להמיר — הטקסט כבר אחיד')

# ── Perform unification ──
cleaned_text = cleaned_text.replace('"', "''")

# ── Verify after unification ──
dq_after   = cleaned_text.count('"')
apos_after = cleaned_text.count("''")

hprint_table('מצב אחרי איחוד', [
    ['סימן', 'קוד', 'מספר'],
    ["''", '2\u00d7U+0027', f'{apos_after}'],
    ['"',  'U+0022',   f'{dq_after}'],
])

# ── Sanity check: known abbreviations exist in text ──
known_rt = {
    "רש''י":     'רבי שלמה יצחקי',
    "הרמב''ם":  'רבי משה בן מימון',
    "הרא''ש":   'רבינו אשר בן יחיאל',
    "מהרי''ל":  'מהר״ם מרוטנבורג',
    "עכו''ם":   'עובדי כוכבים ומזלות',
    "הרשב''א":  'רבי שלמה בן אדרת',
    "סמ''ג":    'ספר מצוות גדול',
    "ר''ן":     'רבינו ניסים',
}

sanity_rows = [['ראשי תיבות', 'מי זה', 'מופעים', 'תקין']]
all_ok = True
for rt, who in known_rt.items():
    cnt = cleaned_text.count(rt)
    ok = '✅' if cnt > 0 else '❌'
    if cnt == 0: all_ok = False
    sanity_rows.append([rt, who, str(cnt), ok])
hprint_table('בדיקת שפיות — ראשי תיבות מוכרים', sanity_rows)

# ── Summary ──
if dq_after == 0 and all_ok:
    hprint(f'✅ איחוד הושלם: {dq_before} מרכאות הומרו ל-\'\', כל הר"ת תקינים')
elif dq_after == 0:
    hprint(f'⚠ המרה בוצעה אך חלק מהר"ת לא נמצאו')
else:
    hprint(f'❌ נותרו {dq_after} מרכאות שלא הומרו!')

סימן,קוד,מספר,אחוז
'',2×U+0027,2117,99.4%
"""",U+0022,13,0.6%


מקור,מספר,תוצאה
"בלע""ז",3,בלע''ז
"להרמב""ם",2,להרמב''ם
"פילטר""ו",1,פילטר''ו
"ספינ""ה",1,ספינ''ה
"סימיא""ה",1,סימיא''ה
"תק""י",1,תק''י
"ולהראב""ד",1,ולהראב''ד
"וסמ""ג",1,וסמ''ג
"של""ד",1,של''ד
"כ""ו",1,כ''ו


סימן,קוד,מספר
'',2×U+0027,2130
"""",U+0022,0


ראשי תיבות,מי זה,מופעים,תקין
רש''י,רבי שלמה יצחקי,72,✅
הרמב''ם,רבי משה בן מימון,50,✅
הרא''ש,רבינו אשר בן יחיאל,144,✅
מהרי''ל,מהר״ם מרוטנבורג,204,✅
עכו''ם,עובדי כוכבים ומזלות,148,✅
הרשב''א,רבי שלמה בן אדרת,42,✅
סמ''ג,ספר מצוות גדול,56,✅
ר''ן,רבינו ניסים,136,✅


## Sample Check — First 15 Lines

In [8]:
sample_lines = cleaned_text.split('\n')[:15]
html = ('<div dir="rtl" style="text-align:right; font-family:monospace; '
        'font-size:13px; line-height:2; '
        'background:#f5f5f5; padding:12px; border-radius:4px;">')
for i, line in enumerate(sample_lines):
    if line.strip() == '':
        html += f'<div style="color:#999;">{i}: (שורה ריקה)</div>'
    else:
        html += f'<div>{i}: {line}</div>'
html += '</div>'
display(HTML(html))

## Save + Download

In [9]:
with open(CLEANED_PATH, 'w', encoding='utf-8') as f:
    f.write(cleaned_text)
hprint(f'✅ נשמר: {CLEANED_PATH} ({len(cleaned_text):,} תווים)')

try:
    colab_files.download(CLEANED_PATH)
    hprint(f'מוריד: {CLEANED_PATH}')
except Exception as e:
    hprint(f'הורדה לא זמינה (לא ב-Colab?): {e}')
    hprint(f'   הקובץ זמין בנתיב: {CLEANED_PATH}')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>